# Lineær Algebra – Konseptuelle Notater

> **Filosofi:** Fokus på *hva ting betyr geometrisk*, ikke *hvordan man regner det ut*.
> Matematikken er et språk for å beskrive rom, retninger og transformasjoner.

---

## Innhold
1. [Vektorrom og underrom](#1-vektorrom-og-underrom)
2. [Matriser – kolonnevektorer som basis](#2-matriser-–-kolonnevektorer-som-basis)
3. [Koeffisientmatrise, rang og determinant](#3-koeffisientmatrise-rang-og-determinant)
4. [De fire fundamentale underrommene](#4-de-fire-fundamentale-underrommene)
5. [Lineære transformasjoner](#5-lineære-transformasjoner)
6. [Egenverdier og egenvektorer](#6-egenverdier-og-egenvektorer)

In [2]:
# Kjør denne cellen først – importerer alt vi trenger
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

---
## 1. Vektorrom og underrom

### Hva er et vektorrom?

Et **vektorrom** $V$ er et sett av vektorer hvor to operasjoner alltid holder deg *inni* rommet:
- **Addisjon:** $\mathbf{u} + \mathbf{v} \in V$ for alle $\mathbf{u}, \mathbf{v} \in V$
- **Skalarmultiplikasjon:** $c\mathbf{v} \in V$ for alle $c \in \mathbb{R}$, $\mathbf{v} \in V$

**Geometrisk intuisjon:** Et vektorrom er et *lukket univers*. Uansett hva du gjør – strekker, speiler, legger sammen – er du alltid inni rommet.

De viktigste eksemplene er $\mathbb{R}^n$ (alle $n$-dimensjonale reelle vektorer), men vektorrom kan også bestå av funksjoner, matriser, eller polynomer.

---

### Underrom

Et **underrom** $S \subseteq V$ er en delmengde av et vektorrom som *selv er et vektorrom*. Det krever tre ting:

1. $\mathbf{0} \in S$ — nullvektoren er med (ellers kollapser skalarmultiplikasjon)
2. Lukket under addisjon
3. Lukket under skalarmultiplikasjon

**Geometrisk:** Underrom i $\mathbb{R}^3$ er alltid én av:
- $\{\mathbf{0}\}$ — kun origo (0-dimensjonalt)
- En **linje gjennom origo** (1-dimensjonalt)
- Et **plan gjennom origo** (2-dimensjonalt)
- Hele $\mathbb{R}^3$ (3-dimensjonalt)

> ⚠️ **Viktig:** Et plan som *ikke* går gjennom origo er **ikke** et underrom! Det er et *affint rom* (et forskjøvet underrom).

---

### Span og lineær uavhengighet

**Span** av en mengde vektorer $\{\mathbf{v}_1, \ldots, \mathbf{v}_k\}$ er *alle* vektorer du kan lage ved lineærkombinasjoner:
$$\text{span}\{\mathbf{v}_1, \ldots, \mathbf{v}_k\} = \{c_1\mathbf{v}_1 + \cdots + c_k\mathbf{v}_k \mid c_i \in \mathbb{R}\}$$

**Geometrisk:** Span er det minste underrommet som inneholder alle vektorene.

Vektorer er **lineært uavhengige** hvis ingen av dem kan skrives som en lineærkombinasjon av de andre. Geometrisk: de peker i *genuint forskjellige retninger* – ingen er "overflødig".

**Basis** = et sett vektorer som både er:
- lineært uavhengige, og
- span hele rommet

En basis er den *mest effektive beskrivelsen* av et rom – ingen vektor er overflødig, og alle retninger er dekket.

**Dimensjon** = antall vektorer i en basis (alltid det samme, uansett hvilken basis du velger).

In [3]:
# VISUALISERING: Underrom i R^3 – et plan og en linje gjennom origo

fig = go.Figure()

# Plan gjennom origo: z = 0 (xy-planet)
xx, yy = np.meshgrid(np.linspace(-2, 2, 10), np.linspace(-2, 2, 10))
zz = np.zeros_like(xx)
fig.add_trace(go.Surface(
    x=xx, y=yy, z=zz,
    opacity=0.35,
    colorscale=[[0, '#4C72B0'], [1, '#4C72B0']],
    showscale=False,
    name='Plan gjennom origo (underrom)'
))

# Linje gjennom origo: retning (1, 1, 1)
t = np.linspace(-2, 2, 50)
fig.add_trace(go.Scatter3d(
    x=t, y=t, z=t,
    mode='lines',
    line=dict(color='#DD4949', width=5),
    name='Linje gjennom origo (underrom)'
))

# Origo
fig.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0],
    mode='markers',
    marker=dict(size=6, color='black'),
    name='Origo'
))

# Plan som IKKE går gjennom origo (ikke et underrom)
zz_shifted = np.ones_like(xx) * 1.2
fig.add_trace(go.Surface(
    x=xx, y=yy, z=zz_shifted,
    opacity=0.25,
    colorscale=[[0, '#888888'], [1, '#888888']],
    showscale=False,
    name='Forskjøvet plan (IKKE underrom)'
))

fig.update_layout(
    title='Underrom i R³ – roter figuren!',
    scene=dict(
        xaxis_title='x', yaxis_title='y', zaxis_title='z',
        aspectmode='cube'
    ),
    legend=dict(x=0, y=1),
    height=550
)
fig.show()

---
## 2. Matriser – kolonnevektorer som basis

### Hva *er* egentlig en matrise?

En matrise kan leses på mange måter. Den mest geometrisk fruktbare:

> **En $m \times n$-matrise er en samling av $n$ kolonnevektorer i $\mathbb{R}^m$.**

$$A = \begin{bmatrix} | & | & & | \\ \mathbf{a}_1 & \mathbf{a}_2 & \cdots & \mathbf{a}_n \\ | & | & & | \end{bmatrix}$$

Kolonnevektorene forteller oss: *"Disse retningene spenner ut bildet av matrisen."*

---

### Matrise-vektor-produkt som lineærkombinasjon

Når vi regner $A\mathbf{x}$, kan vi tenke på det slik:

$$A\mathbf{x} = x_1\mathbf{a}_1 + x_2\mathbf{a}_2 + \cdots + x_n\mathbf{a}_n$$

**$\mathbf{x}$ er koeffisientene i en lineærkombinasjon av kolonnene!**

Dette betyr at $A\mathbf{x} = \mathbf{b}$ har en løsning *hvis og bare hvis* $\mathbf{b}$ kan skrives som en lineærkombinasjon av kolonnene i $A$ – altså hvis $\mathbf{b} \in \text{kolonnerom}(A)$.

---

### Standardbasis og basisskifte

Standardbasen i $\mathbb{R}^2$ er $\mathbf{e}_1 = \begin{bmatrix}1\\0\end{bmatrix}$, $\mathbf{e}_2 = \begin{bmatrix}0\\1\end{bmatrix}$.

Enhver vektor $\mathbf{v} = \begin{bmatrix}3\\2\end{bmatrix}$ betyr egentlig: *"3 skritt langs $\mathbf{e}_1$, 2 skritt langs $\mathbf{e}_2$"*.

Men vi *kan velge andre basisvektorer*. Hvis vi velger $\mathbf{b}_1 = \begin{bmatrix}2\\1\end{bmatrix}$, $\mathbf{b}_2 = \begin{bmatrix}0\\1\end{bmatrix}$, så beskrives den samme vektoren med *andre koordinater* – men den peker fortsatt samme sted i rommet.

> Koordinater er *ikke* vektoren selv – de er oppskriften på å nå vektoren *i en gitt basis*.

In [4]:
# VISUALISERING: Kolonnerom til en 3x2-matrise – et plan i R^3
# Kolonnene a1, a2 spenner ut et plan (kolonnerommet)

a1 = np.array([1, 0, 1])
a2 = np.array([0, 1, 1])

# Spenner ut planet med lineærkombinasjoner
s = np.linspace(-2, 2, 15)
t = np.linspace(-2, 2, 15)
S, T = np.meshgrid(s, t)
X = S * a1[0] + T * a2[0]
Y = S * a1[1] + T * a2[1]
Z = S * a1[2] + T * a2[2]

fig = go.Figure()

# Kolonnerommet (planet)
fig.add_trace(go.Surface(
    x=X, y=Y, z=Z,
    opacity=0.4,
    colorscale=[[0, '#4C72B0'], [1, '#4C72B0']],
    showscale=False,
    name='Kolonnerom C(A)'
))

# Kolonnevektorene
for vec, color, label in [
    (a1, '#DD4949', 'a₁ = (1,0,1)'),
    (a2, '#2CA02C', 'a₂ = (0,1,1)')
]:
    fig.add_trace(go.Scatter3d(
        x=[0, vec[0]], y=[0, vec[1]], z=[0, vec[2]],
        mode='lines+markers+text',
        line=dict(color=color, width=6),
        marker=dict(size=[2, 7], color=color),
        text=['', label],
        textposition='top center',
        name=label
    ))

# En vektor i kolonnerommet (lineærkombinasjon)
combo = 1.2 * a1 + 0.8 * a2
fig.add_trace(go.Scatter3d(
    x=[0, combo[0]], y=[0, combo[1]], z=[0, combo[2]],
    mode='lines+markers+text',
    line=dict(color='#FF7F0E', width=5, dash='dash'),
    marker=dict(size=[2, 7], color='#FF7F0E'),
    text=['', '1.2·a₁ + 0.8·a₂'],
    textposition='top center',
    name='Lineærkombinasjon (i kolonnerom)'
))

fig.update_layout(
    title='Kolonnerom til A = [a₁ | a₂] – et plan i R³',
    scene=dict(
        xaxis_title='x', yaxis_title='y', zaxis_title='z',
        aspectmode='cube'
    ),
    height=580
)
fig.show()

---
## 3. Koeffisientmatrise, rang og determinant

### Koeffisientmatrisen og likningssystemer

Systemet $A\mathbf{x} = \mathbf{b}$ samler alle lineære likninger i én matrise. Kolonnerom-perspektivet gir oss et svar på *når* systemet har løsning:

| Situasjon | Geometrisk | Løsninger |
|-----------|------------|----------|
| $\mathbf{b} \in C(A)$ | $\mathbf{b}$ ligger i kolonnerommet | Minst én løsning |
| $\mathbf{b} \notin C(A)$ | $\mathbf{b}$ er utenfor kolonnerommet | Ingen løsning |
| $N(A) \neq \{\mathbf{0}\}$ | Nullrommet er ikke-trivielt | Uendelig mange løsninger |

---

### Rang

**Rangen** $r = \text{rank}(A)$ er dimensjonen til kolonnerommet – altså *antall lineært uavhengige kolonner*.

Geometrisk: hvor mange genuint ulike retninger strekker kolonnevektorene seg i?

- Hvis $A$ er $3 \times 3$ og $\text{rank}(A) = 3$: kolonnene spenner ut hele $\mathbb{R}^3$
- Hvis $\text{rank}(A) = 2$: kolonnene spenner bare ut et plan
- Hvis $\text{rank}(A) = 1$: alle kolonner peker langs én linje

**Rang-nullitet-teoremet** (fundamentalt!):
$$\text{rank}(A) + \text{nullitet}(A) = n$$
der $n$ er antall kolonner. Det som "går tapt" i transformasjonen (nullrommet) kompenserer nøyaktig for det som "overlever" (kolonnerommet).

---

### Determinant

**Determinanten** $\det(A)$ måler *volumet av det parallelleplipedet som kolonnene danner*.

For $2 \times 2$: $\det(A)$ er arealet av parallelogrammet dannet av de to kolonnevektorene (med fortegn).
For $3 \times 3$: $\det(A)$ er volumet av parallelleplipedet.

Geometrisk betydning av fortegnet: positiv determinant betyr at kolonnene danner et *høyreorientert* system, negativ betyr *venstreorientert* (speiling).

**Den viktigste egenskapen:** $\det(A) = 0 \iff A$ er singulær $\iff$ kolonnene er lineært avhengige $\iff$ kolonnerommet har lavere dimensjon enn forventet.

Geometrisk: volumet kollapser til null fordi kolonnene alle ligger i et lavere-dimensjonalt rom (f.eks. tre vektorer som alle ligger i et plan).

In [5]:
# VISUALISERING: Determinant som areal/volum

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('det ≠ 0: Kolonnene spenner ut areal', 'det = 0: Kolonnene er parallelle'),
    specs=[[{'type': 'xy'}, {'type': 'xy'}]]
)

def draw_parallelogram(fig, v1, v2, row, col, color_fill, label1, label2):
    # Parallelogrammet
    px_vals = [0, v1[0], v1[0]+v2[0], v2[0], 0]
    py_vals = [0, v1[1], v1[1]+v2[1], v2[1], 0]
    fig.add_trace(go.Scatter(x=px_vals, y=py_vals, fill='toself',
        fillcolor=color_fill, line=dict(color='gray', width=1),
        name='Parallelogram', showlegend=False), row=row, col=col)
    # Vektorene
    for v, color, label in [(v1, '#DD4949', label1), (v2, '#4C72B0', label2)]:
        fig.add_trace(go.Scatter(
            x=[0, v[0]], y=[0, v[1]],
            mode='lines+markers+text',
            line=dict(color=color, width=3),
            marker=dict(size=[2, 10], color=color, symbol=['circle', 'arrow'], angleref='previous'),
            text=['', label], textposition='top right',
            showlegend=False
        ), row=row, col=col)

# Venstre: ikke-singulær
draw_parallelogram(fig, np.array([2, 0.5]), np.array([0.5, 2]),
                   1, 1, 'rgba(76, 114, 176, 0.3)', 'a₁', 'a₂')

# Høyre: singulær (parallelle vektorer)
draw_parallelogram(fig, np.array([2, 1]), np.array([1.4, 0.7]),
                   1, 2, 'rgba(200, 80, 80, 0.15)', 'a₁', 'a₂ ≈ 0.7·a₁')

fig.update_xaxes(range=[-0.5, 3.5], zeroline=True, zerolinewidth=1)
fig.update_yaxes(range=[-0.5, 3.0], zeroline=True, zerolinewidth=1, scaleanchor='x', scaleratio=1)
fig.update_layout(height=400, title_text='Determinant som areal')
fig.show()

---
## 4. De fire fundamentale underrommene

Dette er kanskje den mest elegante strukturen i lineær algebra. Enhver $m \times n$-matrise $A$ definerer *fire underrom* – to i $\mathbb{R}^n$ (inputsiden) og to i $\mathbb{R}^m$ (outputsiden).

### Oversikt

| Rom | Symbol | Definisjon | Dimensjon | Hvor |
|-----|--------|-----------|-----------|------|
| Kolonnerom | $C(A)$ | Alle $A\mathbf{x}$ | $r$ | $\mathbb{R}^m$ |
| Nullrom | $N(A)$ | Alle $\mathbf{x}$: $A\mathbf{x}=\mathbf{0}$ | $n-r$ | $\mathbb{R}^n$ |
| Radrom | $C(A^T)$ | Alle $A^T\mathbf{y}$ | $r$ | $\mathbb{R}^n$ |
| Venstre nullrom | $N(A^T)$ | Alle $\mathbf{y}$: $A^T\mathbf{y}=\mathbf{0}$ | $m-r$ | $\mathbb{R}^m$ |

### Den store symmetrien

De fire rommene splitter seg i to ortogonakomplementpar:

$$C(A^T) \perp N(A) \quad \text{i } \mathbb{R}^n$$
$$C(A) \perp N(A^T) \quad \text{i } \mathbb{R}^m$$

Dette betyr at inputsiden $\mathbb{R}^n$ er delt inn i to ortogonale rom:
- **Radrommmet** $C(A^T)$: vektorer som "overlever" gjennom $A$ – de bidrar til output
- **Nullrommet** $N(A)$: vektorer som "drukner" – $A$ sender dem til $\mathbf{0}$

Tilsvarende deles outputsiden $\mathbb{R}^m$:
- **Kolonnerommet** $C(A)$: vektorer $A$ *kan* treffe
- **Venstre nullrommet** $N(A^T)$: vektorer $A$ *aldri* kan treffe

### Geometrisk bilde

Tenk på $A$ som en maskin: input-rommet $\mathbb{R}^n$ er delt i to ortogonale biter. Nullrommet slettes. Radrommmet bevares og mappes *injektivt* over til kolonnerommet. Venstre nullrommet er "blinde flekken" – ingen input når dit.

```
        R^n                      R^m
  ┌─────────────┐          ┌─────────────┐
  │  Radrom     │ ─── A ──▶│ Kolonnerom  │
  │  C(A^T)     │          │  C(A)       │
  ├─────────────┤          ├─────────────┤
  │  Nullrom    │ ─── A ──▶│ {0}         │
  │  N(A)       │          │ (venstre    │
  └─────────────┘          │  nullrom)   │
                           └─────────────┘
```

In [6]:
# VISUALISERING: De fire fundamentale underrommene for en 3x2-matrise
# A er 3x2 med rang 1: kolonner er parallelle
# Kolonnerom: 1D (linje i R^3), Nullrom: 1D (linje i R^2)
# Her bruker vi en 3x2 matrise med rang 1 for å se alt tydelig

A = np.array([[1, 2], [2, 4], [1, 2]], dtype=float)  # rang 1
# Kolonnerom: span av (1,2,1) i R^3
# Nullrom: span av (2,-1) i R^2 (siden 2*a1 - 1*a2 = 0... nei, a2 = 2*a1)
# Nullrom: løsning til x1 + 2*x2 = 0 → x = t*(-2, 1)

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'scene'}, {'type': 'xy'}]],
    subplot_titles=(
        'R³ (outputside): Kolonnerom (blå linje) & Venstre nullrom (grå plan)',
        'R² (inputside): Radrom (rød) & Nullrom (grønn) – ortogonale!'
    )
)

t = np.linspace(-2.5, 2.5, 60)

# Kolonnerom: linje langs (1,2,1)
col_dir = np.array([1, 2, 1]) / np.linalg.norm([1, 2, 1])
fig.add_trace(go.Scatter3d(
    x=t*col_dir[0], y=t*col_dir[1], z=t*col_dir[2],
    mode='lines', line=dict(color='#4C72B0', width=6),
    name='Kolonnerom C(A)'
), row=1, col=1)

# Venstre nullrom: ortogonalt komplement til kolonnerommet i R^3
# Alle v slik at (1,2,1)·v = 0 → et plan
s_vals = np.linspace(-2, 2, 10)
u_vals = np.linspace(-2, 2, 10)
SV, UV = np.meshgrid(s_vals, u_vals)
# Grunnvektorer i planet: (2,-1,0) og (1,0,-1)
b1 = np.array([2, -1, 0]) / np.linalg.norm([2, -1, 0])
b2 = np.array([1, 0, -1]) / np.linalg.norm([1, 0, -1])
PX = SV * b1[0] + UV * b2[0]
PY = SV * b1[1] + UV * b2[1]
PZ = SV * b1[2] + UV * b2[2]
fig.add_trace(go.Surface(
    x=PX, y=PY, z=PZ,
    opacity=0.25,
    colorscale=[[0, '#888888'], [1, '#888888']],
    showscale=False,
    name='Venstre nullrom N(Aᵀ)'
), row=1, col=1)

# Inputside (R²): Radrom og Nullrom
# Radrom: span av (1,2) (første rad)
# Nullrom: span av (-2,1) – orthogonal til (1,2)
t2 = np.linspace(-2.5, 2.5, 60)
radrom_dir = np.array([1, 2]) / np.linalg.norm([1, 2])
nullrom_dir = np.array([-2, 1]) / np.linalg.norm([-2, 1])

fig.add_trace(go.Scatter(
    x=t2*radrom_dir[0], y=t2*radrom_dir[1],
    mode='lines', line=dict(color='#DD4949', width=4),
    name='Radrom C(Aᵀ)'
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=t2*nullrom_dir[0], y=t2*nullrom_dir[1],
    mode='lines', line=dict(color='#2CA02C', width=4),
    name='Nullrom N(A)'
), row=1, col=2)

fig.update_layout(
    height=520,
    title='De fire fundamentale underrommene (rang-1 matrise)',
    scene=dict(
        xaxis_title='x', yaxis_title='y', zaxis_title='z',
        aspectmode='cube'
    )
)
fig.update_xaxes(range=[-2.5, 2.5], zeroline=True, row=1, col=2)
fig.update_yaxes(range=[-2.5, 2.5], zeroline=True, scaleanchor='x', scaleratio=1, row=1, col=2)
fig.show()

---
## 5. Lineære transformasjoner

### Hva er en lineær transformasjon?

En funksjon $T: \mathbb{R}^n \to \mathbb{R}^m$ er *lineær* hvis:
$$T(\mathbf{u} + \mathbf{v}) = T(\mathbf{u}) + T(\mathbf{v})$$
$$T(c\mathbf{v}) = cT(\mathbf{v})$$

Geometrisk: $T$ bevarer *rutenettstruktur*. Rette linjer forblir rette, parallelle linjer forblir parallelle, og origo flyttes ikke.

Enhver lineær transformasjon kan beskrives av en matrise: $T(\mathbf{x}) = A\mathbf{x}$.

### Viktige geometriske transformasjoner i $\mathbb{R}^2$

| Transformasjon | Matrise | Effekt |
|---------------|---------|--------|
| Rotasjon $\theta$ | $\begin{bmatrix}\cos\theta & -\sin\theta \\ \sin\theta & \cos\theta\end{bmatrix}$ | Roterer rundt origo |
| Speiling (x-akse) | $\begin{bmatrix}1 & 0 \\ 0 & -1\end{bmatrix}$ | Speiler om x-aksen |
| Projeksjon (x-akse) | $\begin{bmatrix}1 & 0 \\ 0 & 0\end{bmatrix}$ | Klemmer ned på x-aksen |
| Shear | $\begin{bmatrix}1 & k \\ 0 & 1\end{bmatrix}$ | Skjærer rommet skjevt |

### Sammensetning er matrisemultiplikasjon

Å bruke $T_1$ etterfulgt av $T_2$ tilsvarer $A_2 A_1 \mathbf{x}$. Rekkefølgen er viktig – matrisemultiplikasjon er generelt ikke kommutativt.

**Geometrisk:** Roter 90° og speil er ikke det samme som speil og roter 90°.

In [7]:
# VISUALISERING: Lineær transformasjon – hva skjer med et rutenett?

def apply_transform(M, points):
    return (M @ points.T).T

# Rutenett
lines_x = [np.array([[x, y] for y in np.linspace(-2, 2, 50)]) for x in np.linspace(-2, 2, 9)]
lines_y = [np.array([[x, y] for x in np.linspace(-2, 2, 50)]) for y in np.linspace(-2, 2, 9)]

theta = np.pi / 4  # 45 grader
transforms = {
    'Rotasjon 45°': np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]]),
    'Shear': np.array([[1, 0.8], [0, 1]]),
    'Projeksjon': np.array([[1, 0], [0, 0.0001]]),  # nesten ned til x-akse
}

fig = make_subplots(rows=1, cols=3,
    subplot_titles=list(transforms.keys()))

for col_idx, (name, M) in enumerate(transforms.items(), 1):
    for lines, color in [(lines_x, 'rgba(76,114,176,0.5)'), (lines_y, 'rgba(221,73,73,0.5)')]:
        for line in lines:
            tl = apply_transform(M, line)
            fig.add_trace(go.Scatter(
                x=tl[:, 0], y=tl[:, 1],
                mode='lines', line=dict(color=color, width=1.2),
                showlegend=False
            ), row=1, col=col_idx)

for col_idx in [1, 2, 3]:
    fig.update_xaxes(range=[-3, 3], zeroline=True, zerolinewidth=1.5, row=1, col=col_idx)
    fig.update_yaxes(range=[-3, 3], zeroline=True, zerolinewidth=1.5,
                     scaleanchor=f'x{"" if col_idx==1 else col_idx}',
                     scaleratio=1, row=1, col=col_idx)

fig.update_layout(height=420, title_text='Lineære transformasjoner – hva skjer med rutenettet?')
fig.show()

---
## 6. Egenverdier og egenvektorer

### Den grunnleggende ideen

For de fleste vektorer vil en lineær transformasjon $A$ *endre retning*. Men noen spesielle vektorer – **egenvektorene** – peker i *samme retning* etter transformasjonen (eller stikk motsatt):

$$A\mathbf{v} = \lambda \mathbf{v}$$

Her er $\mathbf{v}$ en egenvektor og $\lambda$ tilhørende **egenverdi** (et tall).

Geometrisk: egenvektorer er de *aksene* som transformasjonen simpelthen strekker eller komprimerer langs, uten å rotere.

### Hva forteller egenverdiene oss?

| $\lambda$ | Effekt på eigenvector |
|-----------|----------------------|
| $\lambda > 1$ | Strekkes ut |
| $0 < \lambda < 1$ | Komprimeres |
| $\lambda = 1$ | Uendret |
| $\lambda = 0$ | Kollapser til $\mathbf{0}$ (vektoren er i nullrommet!) |
| $\lambda < 0$ | Snur retning + skalerer |

### Koblingen til de fire underrommene

- Hvis $\lambda = 0$ er en egenverdi, betyr det at $A\mathbf{v} = \mathbf{0}$ for en ikke-null $\mathbf{v}$ – altså er $\mathbf{v}$ i nullrommet til $A$.
- Dette er ekvivalent med at $A$ er singulær og $\det(A) = 0$.

### Diagonalisering og hvorfor det er nyttig

Hvis $A$ har $n$ lineært uavhengige egenvektorer, kan vi skrive:
$$A = PDP^{-1}$$
der $P$ er matrisen med egenvektorene som kolonner, og $D$ er diagonal med egenverdiene.

Geometrisk: vi skifter til et koordinatsystem der transformasjonen *bare strekker langs aksene*. Det er alltid lettere å tenke i egenvektorbasisen.

> **Intuisjon:** Enhver lineær transformasjon ser *enkel* ut hvis du bare velger riktig basis å se den fra.

In [8]:
# VISUALISERING: Egenvektorer – vektorer som bare skaleres, ikke roteres

A = np.array([[3, 1], [0, 2]], dtype=float)
eigenvalues, eigenvectors = np.linalg.eig(A)

fig = go.Figure()

# Vis et knippe vektorer og deres bilder under A
angles = np.linspace(0, 2*np.pi, 16, endpoint=False)
for angle in angles:
    v = np.array([np.cos(angle), np.sin(angle)])
    Av = A @ v
    fig.add_trace(go.Scatter(
        x=[0, v[0]], y=[0, v[1]],
        mode='lines', line=dict(color='rgba(150,150,150,0.4)', width=1.5),
        showlegend=False
    ))
    fig.add_trace(go.Scatter(
        x=[0, Av[0]], y=[0, Av[1]],
        mode='lines', line=dict(color='rgba(76,114,176,0.5)', width=1.5),
        showlegend=False
    ))

# Egenvektorene (normaliserte)
colors = ['#DD4949', '#2CA02C']
for i, (ev, lam) in enumerate(zip(eigenvectors.T, eigenvalues)):
    ev_n = ev / np.linalg.norm(ev)
    Aev = A @ ev_n
    fig.add_trace(go.Scatter(
        x=[0, ev_n[0]], y=[0, ev_n[1]],
        mode='lines+markers', line=dict(color=colors[i], width=3),
        marker=dict(size=[0, 10], color=colors[i]),
        name=f'Egenvektor v{i+1} (λ={lam:.1f})',
    ))
    fig.add_trace(go.Scatter(
        x=[0, Aev[0]], y=[0, Aev[1]],
        mode='lines+markers', line=dict(color=colors[i], width=3, dash='dot'),
        marker=dict(size=[0, 12], color=colors[i], symbol='star'),
        name=f'A·v{i+1} = {lam:.1f}·v{i+1}',
    ))

fig.update_layout(
    title='Egenvektorer: kun skalert, ikke rotert (grå=input, blå=output)',
    xaxis=dict(range=[-4, 4], zeroline=True, zerolinewidth=1.5),
    yaxis=dict(range=[-4, 4], zeroline=True, zerolinewidth=1.5, scaleanchor='x', scaleratio=1),
    height=480
)
fig.show()

---
## Sammenhenger mellom konseptene

```
Matrise A (m×n, rang r)
        │
        ├─── Kolonnevektorer ──▶ Kolonnerom C(A)  ──┐
        │                          (dim = r)        │  Ortogonale
        │                                           │  komplementer
        ├─── Radreduksjon ─────▶ Rang r             │  i R^m
        │                                           │
        ├─── Ax=0 ─────────────▶ Nullrom N(A)  ◀──┘
        │                          (dim = n-r)
        │
        ├─── Rang-nullitet: r + (n-r) = n  ✓
        │
        ├─── det(A) = 0  ⟺  r < n  ⟺  N(A) ≠ {0}
        │
        └─── Egenvektorer = aksene der A kun skalerer
             λ = 0  ⟺  egenvektor er i nullrommet
```

---
*Notater av Fredrik – konseptuell og geometrisk tilnærming til lineær algebra.*